# Post-processing and standard plots

This tutorial introduces the standardized post-processing interface. We run a small Vlasov–Ampère example, get its output as an autocomplete-friendly `Output`, and make the plots most commonly used to inspect a simulation.

For a production run you can skip the simulation setup and open its output folder with `struphy.Output("path/to/sim")` instead.

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np

from IPython.display import HTML


from struphy import (
    BinningPlot,
    BoundaryParameters,
    ButcherTableau,
    DerhamOptions,
    EnvironmentOptions,
    KernelDensityPlot,
    LoadingParameters,
    SavingParameters,
    Simulation,
    SortingParameters,
    Time,
    WeightsParameters,
    domains,
    equils,
    grids,
    maxwellians,
    perturbations,
)
from struphy.models import Maxwell, ViscousEulerSPH, VlasovAmpereOneSpecies

## Create a compact demonstration run

Post-processing operates on a completed run. The small setup below saves an electric field, a few marker trajectories, scalar diagnostics, and a binned $(\eta_1,v_1)$ distribution. These are the main output types handled by the plotting interface.

In [ ]:
def build_model():
    model = VlasovAmpereOneSpecies(alpha=1.0, epsilon=-1.0, with_B0=False)
    model.em_fields.e_field.save_data = True
    model.em_fields.phi.save_data = True
    model.kinetic_ions.var.save_data = True

    model.propagators.push_eta.options = model.propagators.push_eta.Options()
    model.propagators.coupling_va.options = model.propagators.coupling_va.Options()
    model.initial_poisson.options = model.initial_poisson.Options(stab_mat="M0")

    binplot = BinningPlot(
        slice="e1_v1",
        n_bins=(32, 32),
        ranges=((0.0, 1.0), (-5.0, 5.0)),
    )
    model.kinetic_ions.set_markers(
        loading_params=LoadingParameters(ppc=32, seed=1234),
        weights_params=WeightsParameters(control_variate=True),
        boundary_params=BoundaryParameters(),
        sorting_params=SortingParameters(boxes_per_dim=(4, 1, 1), do_sort=True),
        saving_params=SavingParameters(n_markers=12, binning_plots=(binplot,)),
    )

    background = maxwellians.Maxwellian3D(n=(1.0, None))
    model.kinetic_ions.var.add_background(background)
    density_mode = perturbations.ModesCos(ls=(1,), amps=(1e-3,))
    model.kinetic_ions.var.add_initial_condition(maxwellians.Maxwellian3D(n=(1.0, density_mode)))

    return model


model = build_model()

In [ ]:
demo_tmp = tempfile.TemporaryDirectory(prefix="struphy_postprocessing_")
demo_root = demo_tmp.name

env = EnvironmentOptions(
    out_folders=demo_root,
    sim_folder="vlasov_ampere_demo",
    save_restart=False,
)
sim = Simulation(
    model=model,
    env=env,
    time_opts=Time(dt=0.05, Tend=5.0),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(16, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
out = sim.run(profiling_activated=True)
print(f"Raw output: {sim.env.path_out}")

## Process and load the output

`sim.run()` returns an `Output`, which is also available later as `sim.output`. Scalars are read directly from the raw output; fields and particle products need post-processing, which runs with default options the first time they are accessed.

To choose options, call `out.pproc()` first. `Output` evaluates saved FEEC fields and organizes particle diagnostics; `physical=True` additionally creates physical field components. Existing products made with the same options are reused, so re-running a cell is cheap.

Individual products are standard `xarray.DataArray` objects with named dimensions, coordinates, units, and labels. Time is in Struphy units, in which the models' analytic results are written; seconds come along as the coordinate `t_seconds`, and `out.with_time_units("physical")` gives an independent view with `t` in seconds. Arrays are loaded only when accessed. The saved configuration is available through `out.domain`, `out.model`, and `out.time_opts`; no simulation object is created.

In [ ]:
out.pproc(physical=True)

Products sit on the run under the species that produced them, so VS Code and interactive shells complete them as you type: `out.kinetic_ions.e1_v1_density.f`, `out.kinetic_ions.orbits`, `out.em_fields.phi_log`. To discover the products a particular namespace contains, inspect its flat, lazy `.catalog`: `list(out.kinetic_ions.catalog)`. The grouped views `out.fields`, `out.distributions`, `out.densities` and `out.orbits` show the same products by kind. In scripts, `out.evaluate("kinetic_ions/f", dataset="e1_v1_density/f")` looks up one explicitly.

In [ ]:
out.info()

In [ ]:
print(out.kinetic_ions)

### Reconstructed setup and initial conditions

`out.info()` prints every evaluable key with a short description and the exact `out.evaluate(...)` call that loads it, followed by a few hints on selecting times and coordinates. The run configuration is not part of `info()`: in `out.metadata`, serialized initial conditions live on each variable under `model → species → variables → initial_conditions`. `out.initial_conditions` gives reconstructed objects, including saved Python functions and classes; `out.model` restores them onto the model variables. Unsupported definitions remain available in `out.metadata`.

In [ ]:
print("Model parameters:", out.model.params)
print("Kinetic variables:", out.model.kinetic_ions.variables)
print("Propagator options:")
for name, options in out.metadata["model"]["propagator_options"].items():
    print(f"  {name}: {options}")

saved = out.metadata["model"]["species"]["kinetic_ions"]["variables"]["var"]["initial_conditions"]
print("Initial-condition entries:", tuple(saved))
initial = out.initial_conditions["kinetic_ions"]["var"]
print("Background distribution:", initial["backgrounds"])
print("Initial distribution:", initial["initial_condition"])

The reconstructed kinetic distributions are ordinary Struphy background objects. Here we evaluate the saved equilibrium and initial distribution along $\eta_1$; their difference is the density perturbation that seeded the run.

In [ ]:
background = initial["backgrounds"]
initial_distribution = initial["initial_condition"]
eta1 = np.linspace(0.0, 1.0, 256)
zeros = np.zeros_like(eta1)
n_background = np.asarray(background.n(eta1, zeros, zeros))
n_initial = np.asarray(initial_distribution.n(eta1, zeros, zeros))

fig, ax = plt.subplots()
ax.plot(eta1, n_background, label="background density")
ax.plot(eta1, n_initial, label="initial density")
ax.plot(eta1, n_initial - n_background, label="density perturbation")
ax.set(xlabel=r"$\eta_1$", ylabel="density", title="Saved kinetic initial condition")
ax.legend();

In [ ]:
phase_space = out.kinetic_ions.e1_v1_density.f
print(phase_space)

### Evaluate saved splines directly: 1-D, 2-D, and 3-D

`evaluate()` reads saved FEEC coefficients and evaluates the spline directly, without creating a post-processing field. With no `eta` arguments, it evaluates the full simulation grid at its cell centres. For cuts, provide only the logical coordinates that should vary; omitted directions use the midpoint, `0.5`. Scalars, lists, NumPy arrays, and `range` objects can be mixed; every non-scalar input becomes a dimension of the tensor-product result. Thus one varying coordinate makes a 1-D line, two make a 2-D plane, and no coordinates makes a 3-D volume. The result is always an xarray array with logical `e1`/`e2`/`e3` coordinates and mapped physical `X`/`Y`/`Z` coordinates.

A representation conversion is applied after spline evaluation. Its input is inferred from the saved FEEC space, so `representation=` specifies only the target: `"0"`, `"1"`, `"2"`, `"3"`, `"v"`, or `"norm"`. Scalars default to `"0"`; vectors default to `"norm"`.

In [ ]:
# 1-D: a field line; eta2 and eta3 default to their midpoints.
eta1_line = np.linspace(0.0, 1.0, 128)
phi_line = out.evaluate(
    "em_fields/phi",
    eta1=eta1_line,
    t=-1,
)
phi_line.plot()
print(phi_line.dims, phi_line.shape)


For a 2-D plane, vary two coordinates and hold the third fixed. This is useful for a cross-section of a 3-D field even when the simulation was run on a coarser grid: the spline is evaluated at the requested points. For a 3-D volume, vary all three coordinates. Keep volume grids modest, then take a plane or line from the labeled result for plotting or further analysis.

In [ ]:
# 2-D: a logical eta1--eta2 plane; eta3 defaults to its midpoint.
phi_plane = out.evaluate(
    "em_fields/phi",
    eta1=np.linspace(0.0, 1.0, 64),
    eta2=np.linspace(0.0, 1.0, 48),
    t=-1,
)
phi_plane.plot(x="e1", y="e2")

# 3-D: no eta arguments uses the complete simulation grid at cell centres.
phi_volume = out.evaluate("em_fields/phi", t=-1)
print(phi_volume.dims, phi_volume.shape)

# xarray plots a 2-D slice of the volume; choose the mid-plane by coordinate index.
phi_volume.isel(e3=phi_volume.sizes["e3"] // 2).plot(x="e1", y="e2")


### Inspecting `out` itself

Most of what `out` exposes is generated on demand (`__getattr__`, cached properties), so `vars(out)` only shows a handful of private cache slots, not the products or methods. Use `dir(out)` for the flat list IPython's own tab-completion relies on, and `out.info()` for the full product tree. The same applies one level down: `out.model` is not a generic stub but the concrete model class of the run (`VlasovAmpereOneSpecies` here), so `dir(out.model)` and `print(out.model)` already show that model's own parameters directly — no separate `OutputVlasovAmpereOneSpecies`-style class is needed.

In [ ]:
print("vars(out):", vars(out))  # only private cache slots

In [ ]:
print("dir(out):", [name for name in dir(out) if not name.startswith("_")])

In [ ]:
print(out.model)  # the concrete model class, with its own parameters

In [ ]:
print("dir(out.model):", [name for name in dir(out.model) if not name.startswith("_")][:15])

### Products are xarray arrays

A product is an `xarray.DataArray`, so xarray's own plotting already draws it, with the labels and units Struphy stored:

In [ ]:
phase_space.isel(t=-1).plot(x="e1", y="v1")

Use xarray's plotting methods for ordinary one- and two-dimensional output. Select a saved time or spatial slice with `.isel()` or `.sel()` first, then call `.plot()` or `.plot.line()`.

## Scalar overview and time series

Field and particle products are xarray `DataArray` objects; `out.evaluate("scalars")` returns an xarray `Dataset`. `out.evaluate("kinetic_ions/f", dataset="e1_v1_density/f")` looks a product up by name, which suits scripts and loops; attribute access is convenient interactively.

Call `.plot()` for a scalar time series. Use a Matplotlib axes when combining several series or setting plot options.

In [ ]:
out.scalars.electric_energy.plot.line(x="t")

In [ ]:
t_fit = 2.0  # Struphy time units, like every time coordinate of this run
energy = out.scalars.electric_energy.sel(t=slice(0.0, t_fit))
fig, ax = plt.subplots()
energy.plot.line(ax=ax, label="electric energy")
ax.set_yscale("log")
ax.legend()

## Two-dimensional data

Choose the displayed dimensions with `x` and `y`, and pick one value for every other dimension by naming it: `t="last"` (or `"first"`), `t=-1` for a position, and `t=0.35` for the nearest coordinate value. Arrays can also be sliced beforehand with xarray's `.isel()` and `.sel()`. `coords="physical"` draws on the mapped coordinates instead of the logical ones.

In [ ]:
phase_space.isel(t=-1).plot(x="e1", y="v1")

For a compact view of the evolution, select saved times and use xarray faceting.

In [ ]:
phase_space.isel(t=np.linspace(0, phase_space.sizes["t"] - 1, 5, dtype=int)).plot(
    x="e1", y="v1", col="t", col_wrap=5
)

## Selecting saved snapshots

Use `.isel()` for index-based selection and `.sel()` for coordinate-based selection. This keeps selection explicit and works with every xarray operation.

In [ ]:
final_phase_space = phase_space.isel(t=-1)
final_phase_space.plot(x="e1", y="v1")

Saved marker orbits sit under their species as an `xarray.Dataset` with one `(t, marker)` variable per quantity (`x`, `y`, `z`, velocities, `weight`); each variable's `description` attribute says what it is. Select one marker and plot its positions over time.

In [ ]:
orbit = out.kinetic_ions.orbits.isel(marker=0)
orbit[["x", "y", "z"]].to_dataarray("quantity").plot.line(x="t", hue="quantity")

A small collection of explicit snapshots is often more useful in a reproducible notebook than an interactive widget or animation.

In [ ]:
phase_space.isel(t=[0, -1]).plot(x="e1", y="v1", col="t")

In [ ]:
fig, ax = plt.subplots()
phase_space.isel(t=-1).plot(ax=ax, x="e1", y="v1")
fig.savefig(os.path.join(demo_root, "phase_space_final.png"), bbox_inches="tight")

The reconstructed equilibrium is available directly on the output handle for inspection and for model-specific analysis.

In [ ]:
print(out.equil)

## Derived quantities

xarray arithmetic computes derived quantities without special APIs. For example, subtract the first sample to obtain a drift and divide by it to obtain a relative error.

In [ ]:
total_energy = out.scalars.total_energy
energy_drift = total_energy - total_energy.isel(t=0)
energy_error = abs(energy_drift) / abs(total_energy.isel(t=0))
print(f"largest drift of the total energy: {abs(energy_drift).max().item():.3e}")

energy_error.plot.line(x="t")

For custom diagnostics, first select the labeled subset needed for the calculation. Here a space-time field line is retained as an xarray object, ready for NumPy, SciPy, or another analysis package.

In [ ]:
space_time_line = out.evaluate(
    "em_fields/phi", eta1=np.linspace(0.0, 1.0, 64), eta2=0.5, eta3=0.5
)
print(space_time_line.dims, space_time_line.shape)

## Reducing distribution functions

A binned distribution usually has more dimensions than the question needs. Xarray reductions retain the remaining named dimensions, so averaging over `e1`, `e2` and `e3` turns the $(\eta_1, v_1)$ product into $f(v_1, t)$. The mean is uniform in logical coordinates, which is a volume average on a Cartesian domain.

In [ ]:
f_of_v = phase_space.mean(("e1", "e2", "e3"), missing_dims="ignore")
print(f_of_v.dims)
f_of_v.plot(x="t", y="v1")

Velocity moments are weighted xarray reductions. The bin widths and velocity coordinate remain labeled, making the density, mean velocity, and variance explicit.

In [ ]:
dv1 = phase_space.v1.differentiate("v1")
density = (phase_space * dv1).sum("v1")
mean_v1 = (phase_space * phase_space.v1 * dv1).sum("v1") / density
variance_v1 = (phase_space * (phase_space.v1 - mean_v1) ** 2 * dv1).sum("v1") / density
mean_density = density.mean(("e1", "e2", "e3"), missing_dims="ignore")
mean_density.plot.line(x="t")
variance_v1.mean(("e1", "e2", "e3"), missing_dims="ignore").plot.line(x="t")

## Physical units

Products are in the normalization of the model. `out.units` holds the units of that normalization and `out.to_si(product)` converts a product: the time coordinate to seconds, the mapped coordinates `X`, `Y`, `Z` to meters and the velocities `v1`, `v2`, `v3` to m/s. The values are converted only when `unit=` names the unit the variable was normalized with (`"x"`, `"B"`, `"n"`, `"v"`, `"t"`, `"p"`, `"rho"`, `"j"` or `"kBT"`), or a number for a composite unit together with its `label`, because a product does not record which unit its variable uses. The original product is not modified.

In [ ]:
print(f"1 length unit = {out.units.x} m, 1 velocity unit = {out.units.v:.4g} m/s, 1 time unit = {out.units.t:.4g} s")

phase_space_si = out.to_si(phase_space)
print(phase_space_si.v1.attrs["units"], phase_space_si.t.attrs["units"])
phase_space_si.isel(t=-1).plot(x="e1", y="v1")

## Save standard output

Every `PlotResult` supports `.save(path)`. For a complete scalar report, `out.save_report()` writes a CSV table, an overview, and one PNG per scalar beneath `post_processing/report/`.

In [ ]:
written = out.save_report()
print("Wrote:")
for path in written:
    print(" ", os.path.relpath(path, out.path_out))

## Comparing runs

Time series accept arrays of other simulations, so comparing runs needs nothing special. Series are labelled by the run they come from, and the runs may have different time grids.

In [ ]:
sim_coarse = Simulation(
    model=build_model(),
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="vlasov_ampere_coarse", save_restart=False),
    time_opts=Time(dt=0.1, Tend=5.0),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(16, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
out_coarse = sim_coarse.run(profiling_activated=True)

fig, ax = plt.subplots()
out.scalars.electric_energy.plot.line(ax=ax, label="dt = 0.05")
out_coarse.scalars.electric_energy.plot.line(ax=ax, label="dt = 0.1")
ax.legend()

## Profiling

A run started with `sim.run(profiling_activated=True)` records how long each region of the code takes, and `out.profile` reads that record back. Regions are the setup steps, every propagator (`prop: ...`), pusher, accumulation, compiled kernel (`kernel: ...`) and linear solve. Regions nest, so a region's time includes the regions it calls and the times of different regions must not be added up. `summary()` returns a dataset along the dimension `region`, sorted by total time; `table()` prints it. Filter with `prefix` and limit with `top`.

In [ ]:
print(out.profile.table(top=8))

kernels = out.profile.summary(prefix="kernel:")
print(kernels.total_time.to_series())

`compare()` puts the same statistic of several runs side by side, with runs whose region is missing as NaN. Here the two runs of the previous section differ only in the time step, so the number of calls per propagator halves for `dt = 0.1`.

In [ ]:
calls = out.profile.compare(out_coarse, metric="calls", prefix="prop:")
print(calls)

## Other models

The interface is the same for every model; only the products differ. Two more short runs show the two product types the Vlasov–Ampère demo does not have: SPH densities, and vector fields on a mapped domain.

### SPH densities

A standing sound wave discretized with SPH markers. `KernelDensityPlot` reconstructs the density on a grid, which appears under `out.densities`, while `BinningPlot` produces the binned quantities under `out.distributions`.

In [ ]:
sph_model = ViscousEulerSPH(with_B0=False, with_viscosity=False)
sph_model.propagators.push_eta.options = sph_model.propagators.push_eta.Options(
    butcher=ButcherTableau(algo="forward_euler"),
)
sph_model.propagators.push_sph_p.options = sph_model.propagators.push_sph_p.Options(kernel_type="gaussian_1d")
sph_model.euler_fluid.set_markers(
    loading_params=LoadingParameters(ppb=8, loading="tesselation"),
    weights_params=WeightsParameters(),
    boundary_params=BoundaryParameters(),
    sorting_params=SortingParameters(boxes_per_dim=(12, 1, 1), dims_mask=(True, False, False)),
    saving_params=SavingParameters(
        binning_plots=(BinningPlot(slice="e1", n_bins=(32,), ranges=(0.0, 1.0)),),
        kernel_density_plots=(KernelDensityPlot(pts_e1=41, pts_e2=1),),
    ),
)
sph_model.euler_fluid.var.add_background(equils.ConstantVelocity())
sph_model.euler_fluid.var.add_perturbation(del_n=perturbations.ModesSin(ls=(1,), amps=(1.0e-2,)))

sph = Simulation(
    model=sph_model,
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="sph_soundwave", save_restart=False),
    time_opts=Time(dt=0.03125, Tend=2.5, split_algo="Strang"),
    domain=domains.Cuboid(r1=2.5),
    grid=None,
    derham_opts=None,
)
out_sph = sph.run()

print("densities:", tuple(out_sph.density_catalog))
print("binned:", tuple(out_sph.distribution_catalog))

For a one-dimensional run, the clearest picture is a space-time map: the sweep dimension `t` may be used as a display axis.

In [ ]:
density = out_sph.euler_fluid.view_0.n.isel(e2=0, e3=0)
density.plot(x="t", y="e1")

Products are plain `xarray.DataArray` objects, so anything xarray can do works directly, for example profiles at selected times:

In [ ]:
density.isel(t=[0, len(density.t) // 4, len(density.t) // 2]).plot.line(x="e1")

### Vector fields on a mapped domain

A coaxial waveguide mode of the Maxwell model, on an annulus. With `physical=True` the post-processing also computes the Cartesian field components (`*_xyz`), and `coords="physical"` draws them on the mapped grid, with the plane chosen by `plane`.

In [ ]:
a1, a2 = 2.326744, 3.686839

maxwell_model = Maxwell()
maxwell_model.propagators.maxwell.options = maxwell_model.propagators.maxwell.Options(algo="implicit")
maxwell_model.em_fields.e_field.add_perturbation(perturbations.CoaxialWaveguideElectric_r(m=3, a1=a1, a2=a2))
maxwell_model.em_fields.e_field.add_perturbation(perturbations.CoaxialWaveguideElectric_theta(m=3, a1=a1, a2=a2))
maxwell_model.em_fields.b_field.add_perturbation(perturbations.CoaxialWaveguideMagnetic(m=3, a1=a1, a2=a2))

coaxial = Simulation(
    model=maxwell_model,
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="coaxial", save_restart=False),
    time_opts=Time(dt=0.05, Tend=2.0),
    domain=domains.HollowCylinder(a1=a1, a2=a2, Lz=2.0),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(24, 48, 1)),
    derham_opts=DerhamOptions(degree=(2, 2, 1), bcs=(("dirichlet", "dirichlet"), None, None)),
)
out_coaxial = coaxial.run()
out_coaxial.pproc(physical=True)

print("fields:", tuple(out_coaxial.field_catalog))
print("dimensions:", out_coaxial.em_fields.b_field_xyz.dims)

In [ ]:
out_coaxial.em_fields.b_field_xyz.isel(t=-1, component=2, e3=0).plot(x="e1", y="e2")

In [ ]:
out_coaxial.em_fields.b_field_xyz.isel(component=2, e3=0).plot(x="e1", y="e2", col="t", col_wrap=4)

### Representation conversion on a torus

A toroidal map makes the distinction between FEEC representations visible. The saved electric field is an H(curl) 1-form, so its source representation is inferred as `1`. We evaluate one poloidal line directly from saved spline coefficients, then request its native 1-form, normalized-vector, and Cartesian-vector representations. These differ away from a Cartesian map because the metric factors vary around the torus.

In [ ]:
torus_model = Maxwell()
torus_model.em_fields.e_field.save_data = True
torus_model.em_fields.e_field.add_perturbation(
    perturbations.ModesCos(ms=(1,), amps=(0.1,), given_in_basis="1", comp=1)
)

torus = Simulation(
    model=torus_model,
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="representation_torus", save_restart=False),
    time_opts=Time(dt=0.05, Tend=0.05),
    domain=domains.HollowTorus(a1=0.2, a2=0.4, R0=1.0, tor_period=1),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(6, 12, 2)),
    derham_opts=DerhamOptions(degree=(2, 2, 2), bcs=(("dirichlet", "dirichlet"), None, None)),
)
out_torus = torus.run()


In [ ]:
eta2_line = np.linspace(0.0, 1.0, 256)
component=0
common = dict(eta1=0.7, eta2=eta2_line, eta3=0.0, t=-1, component=component)
e_1 = out_torus.evaluate("em_fields/e_field", representation="1", **common)
e_norm = out_torus.evaluate("em_fields/e_field", representation="norm", **common)
e_v = out_torus.evaluate("em_fields/e_field", representation="v", **common)

fig, ax = plt.subplots(ncols=3, figsize=(15, 5))
i = 0
for field, label in ((e_1, "1-form"), (e_norm, "normalized vector"), (e_v, "Vector field")):
    print(label, field.isel(t=0))
    ax[i].plot(eta2_line, field.isel(t=0), label=label)
    
    ax[i].set(xlabel=r"$\eta_2$", ylabel=label, title="H(curl) field representations on a torus")
    i += 1
# ax.legend()



## Apply the workflow to another run

For an already completed simulation, possibly in a separate process without MPI, open its output folder:

```python
import struphy

out = struphy.Output("/path/to/sim_1").pproc(physical=True)
out.domain, out.model.units  # reconstructed directly from saved metadata
```

Use `out.scalars`, `out.fields`, `out.distributions`, `out.orbits`, and `out.densities`. Attribute access is the normal interactive API; the corresponding `*_catalog` mappings are intended for generic loops and tooling.